# Rare Disease Genes – Homework 3: Visualization

**Instructions:** Run the Setup cell first, then complete each exercise in order.  
Write your code in the `# YOUR CODE HERE` cells. Answer written questions in the markdown cells below each prompt.

**What you'll learn:**
- Merge two datasets and explore the combined data
- Create bar charts, scatter plots, and grouped bar charts
- Use histograms and box plots to explore distributions
- Build a heatmap to spot patterns across two dimensions
- Investigate disease–gene patterns across chromosomes

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load both datasets
diseases = pd.read_csv('https://raw.githubusercontent.com/narayananr/rare_diseases/main/data/rare_gene_disease_dataset.csv')
genes = pd.read_csv('https://raw.githubusercontent.com/narayananr/rare_diseases/main/data/ensembl_genes.csv')

# Merge them (you learned this in Class 2!)
merged = diseases.merge(genes, on='gene_symbol', how='left')

# Calculate gene length (we'll use this throughout)
genes['length'] = genes['end'] - genes['start']
merged['length'] = merged['end'] - merged['start']

print(f'Diseases dataset: {len(diseases)} rows')
print(f'Genes dataset: {len(genes)} rows')
print(f'Merged dataset: {len(merged)} rows, {merged.shape[1]} columns')
print(f'\nColumns: {list(merged.columns)}')
merged.head()

---
## Part 1: Where Do Disease Genes Live?

Humans have 23 pairs of chromosomes, but are rare disease genes spread evenly across them — or do some chromosomes carry more than their fair share?

**Question: Which chromosomes carry the most rare disease genes?**

In [ ]:
# Step 1: Prepare the data
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT']

disease_by_chrom = (
    merged['chromosome']
    .value_counts()
    .reindex(chrom_order)
    .dropna()
    .reset_index()
)
disease_by_chrom.columns = ['chromosome', 'count']

print("Disease-gene links per chromosome:")
disease_by_chrom

In [ ]:
# Step 2: Plot it
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=disease_by_chrom, x='chromosome', y='count', color='steelblue', ax=ax)
ax.set_xlabel('Chromosome')
ax.set_ylabel('Number of Disease-Gene Links')
ax.set_title('Rare Disease-Gene Links per Chromosome')
plt.tight_layout()
plt.show()

### 🎯 Exercise 1: Does Counting Method Matter?

The chart above counts disease-gene **links** — but one gene can appear many times if it's linked to many diseases. **Does the picture change when we count unique genes instead?**

Create a chart that counts **unique genes** per chromosome.

*Hints:*
- Use `merged.drop_duplicates(subset='gene_symbol')` to get one row per gene
- Then count by chromosome with `.value_counts()` and `.reindex(chrom_order)`
- Use the demo chart above as a template

In [ ]:
# YOUR CODE HERE: Bar chart of unique rare disease genes per chromosome



<details>
<summary>💡 Click here for the solution</summary>

```python
# Step 1: Prepare the data
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT']
unique_genes = merged.drop_duplicates(subset='gene_symbol')
unique_by_chrom = (
    unique_genes['chromosome']
    .value_counts()
    .reindex(chrom_order)
    .dropna()
    .reset_index()
)
unique_by_chrom.columns = ['chromosome', 'count']
unique_by_chrom

# Step 2: Plot it
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=unique_by_chrom, x='chromosome', y='count', color='steelblue', ax=ax)
ax.set_xlabel('Chromosome')
ax.set_ylabel('Number of Unique Rare Disease Genes')
ax.set_title('Unique Rare Disease Genes per Chromosome')
plt.tight_layout()
plt.show()
```

</details>

**Answer these questions** (double-click to edit):

1. Which chromosome has the most unique rare disease genes?  
   *Your answer:*

2. Does chromosome size seem to matter? (Chromosome 1 is the largest, chromosome 21 is one of the smallest.)  
   *Your answer:*

3. How does the X chromosome compare to similarly-sized autosomes?  
   *Your answer:*

### Going Deeper: Is Chromosome 1 Really Special, or Just Big?

The bar chart shows chromosome 1 has the most disease genes — but it's also the **largest** chromosome with the most genes overall. 

**Question: Which chromosomes have a disproportionately high fraction of disease genes?**

To answer this, we calculate **enrichment** — what percentage of each chromosome's genes are linked to rare diseases. This normalizes for chromosome size.

In [ ]:
# Step 1: Prepare the data — calculate % of genes that are disease-related
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

total_per_chrom = genes[genes['chromosome'].isin(chrom_order)].groupby('chromosome').size()
rare_per_chrom = (
    merged.drop_duplicates(subset='gene_symbol')
    .query("chromosome in @chrom_order")
    .groupby('chromosome').size()
)

enrichment_df = pd.DataFrame({
    'chromosome': chrom_order,
    'total_genes': total_per_chrom.reindex(chrom_order).values,
    'rare_disease_genes': rare_per_chrom.reindex(chrom_order).fillna(0).astype(int).values
})
enrichment_df['pct_disease'] = (enrichment_df['rare_disease_genes'] / enrichment_df['total_genes'] * 100).round(1)

print("Disease gene enrichment by chromosome:")
enrichment_df

In [ ]:
# Step 2: Plot it
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=enrichment_df, x='chromosome', y='pct_disease', color='coral', ax=ax)
ax.set_xlabel('Chromosome')
ax.set_ylabel('% of Genes Linked to Rare Diseases')
ax.set_title('Disease Gene Enrichment by Chromosome')
ax.axhline(y=enrichment_df['pct_disease'].mean(), color='gray', linestyle='--', label='Average')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Average across chromosomes: {enrichment_df['pct_disease'].mean():.1f}%")

**Compare the two charts** (double-click to edit):

1. Which chromosome ranks highest in raw count? Which ranks highest in enrichment (%)? Are they the same?  
   *Your answer:*

2. Why is enrichment a more meaningful measure than raw count?  
   *Your answer:*

---
### Which Genes Are the "Weakest Links" in Human Health?

Some genes are linked to many different diseases — when they break, many things go wrong. These **hub genes** are often involved in fundamental biological processes like cell growth or DNA repair.

**Question: If a single gene mutates, how many different diseases could result?**

In [ ]:
# Step 1: Prepare the data — count diseases per gene
hub_genes = (
    diseases['gene_symbol']
    .value_counts()
    .head(15)
    .reset_index()
)
hub_genes.columns = ['gene_symbol', 'disease_count']

print("Top 15 genes linked to the most diseases:")
hub_genes

In [ ]:
# Step 2: Plot it — horizontal bar chart so gene names are readable
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=hub_genes, x='disease_count', y='gene_symbol', color='steelblue', ax=ax)
ax.set_xlabel('Number of Diseases')
ax.set_ylabel('Gene')
ax.set_title('Top 15 "Hub" Genes Linked to Most Rare Diseases')
plt.tight_layout()
plt.show()

**Think about it** (double-click to edit):

1. Look up the #1 gene online. What does it do? Why might it be linked to so many diseases?  
   *Your answer:*

---
### Are Some Diseases Genetically Simple While Others Are Complex?

Some diseases are caused by a single gene mutation — one gene breaks, one disease results. Others are genetically complex — dozens of different genes can cause or contribute to them.

**Question: Which rare diseases involve the most genes?**

In [ ]:
# Step 1: Prepare the data — count genes per disease
complex_diseases = (
    diseases['disease']
    .value_counts()
    .head(15)
    .reset_index()
)
complex_diseases.columns = ['disease', 'gene_count']

print("Top 15 most genetically complex diseases:")
complex_diseases

In [ ]:
# Step 2: Plot it
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=complex_diseases, x='gene_count', y='disease', color='indianred', ax=ax)
ax.set_xlabel('Number of Genes')
ax.set_ylabel('Disease')
ax.set_title('Top 15 Most Genetically Complex Rare Diseases')
plt.tight_layout()
plt.show()

**Think about it** (double-click to edit):

1. What is the most genetically complex disease? How many genes are involved?  
   *Your answer:*

2. Why might some diseases involve many genes while others involve only one?  
   *Your answer:*

---
## Part 2: Are Disease Genes Bigger Than Normal?

Genes vary hugely in size — some are a few hundred base pairs, others span millions. Larger genes make bigger targets for mutations.

**Question: Are rare disease genes longer or shorter than the average human gene?**

In [ ]:
# Gene length is already calculated in the setup cell
unique_disease_genes = merged.drop_duplicates(subset='gene_symbol').copy()

# Compare averages
avg_all = genes['length'].mean()
avg_rare = unique_disease_genes['length'].dropna().mean()

print(f"Average gene length (all genes):          {avg_all:,.0f} base pairs")
print(f"Average gene length (rare disease genes):  {avg_rare:,.0f} base pairs")

### Does Gene Size Vary by Chromosome?

We know the overall average — but do some chromosomes have longer disease genes than others?

**Question: Which chromosomes carry the longest rare disease genes?**

In [ ]:
# Step 1: Prepare the data
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

rare_length_by_chrom = (
    unique_disease_genes
    .groupby('chromosome')['length']
    .mean()
    .reindex(chrom_order)
    .reset_index()
)
rare_length_by_chrom.columns = ['chromosome', 'avg_length']
rare_length_by_chrom['avg_length_kb'] = rare_length_by_chrom['avg_length'] / 1000

print("Average rare disease gene length per chromosome (kb):")
rare_length_by_chrom[['chromosome', 'avg_length_kb']]

In [ ]:
# Step 2: Plot it
fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(data=rare_length_by_chrom, x='chromosome', y='avg_length_kb', color='steelblue', ax=ax)
ax.set_xlabel('Chromosome')
ax.set_ylabel('Average Gene Length (kb)')
ax.set_title('Average Rare Disease Gene Length by Chromosome')
plt.tight_layout()
plt.show()

### 🎯 Exercise 2: Is the Size Difference Consistent Across Chromosomes?

We saw that rare disease genes are longer *on average*. But is that true on every chromosome, or just a few?

**Your task:** Create a grouped bar chart comparing average gene length on each chromosome for all genes vs rare disease genes.

Use the template below — fill in the missing parts.

In [ ]:
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

# Step 1: Prepare the data
avg_all_by_chrom = genes.groupby('chromosome')['length'].mean().reindex(chrom_order)

# YOUR CODE HERE: Average gene length per chromosome for rare disease genes
# Hint: use unique_disease_genes.groupby('chromosome')['length'].mean()
# avg_rare_by_chrom = ???

# Build a DataFrame for plotting
# YOUR CODE HERE: uncomment and fill in avg_rare_by_chrom
# length_df = pd.DataFrame({
#     'chromosome': chrom_order,
#     'All Genes': avg_all_by_chrom.values / 1000,
#     'Rare Disease Genes': avg_rare_by_chrom.values / 1000
# })
# length_melted = length_df.melt(id_vars='chromosome', var_name='group', value_name='avg_length_kb')
# length_df

# Step 2: Plot it
# fig, ax = plt.subplots(figsize=(14, 6))
# sns.barplot(data=length_melted, x='chromosome', y='avg_length_kb', hue='group',
#             palette=['lightcoral', 'steelblue'], ax=ax)
# ax.set_xlabel('Chromosome')
# ax.set_ylabel('Average Gene Length (kb)')
# ax.set_title('Average Gene Length: All Genes vs Rare Disease Genes')
# ax.legend(title='')
# plt.tight_layout()
# plt.show()


<details>
<summary>💡 Click here for the solution</summary>

```python
# Step 1: Prepare the data
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

avg_all_by_chrom = genes.groupby('chromosome')['length'].mean().reindex(chrom_order)
avg_rare_by_chrom = unique_disease_genes.groupby('chromosome')['length'].mean().reindex(chrom_order)

length_df = pd.DataFrame({
    'chromosome': chrom_order,
    'All Genes': avg_all_by_chrom.values / 1000,
    'Rare Disease Genes': avg_rare_by_chrom.values / 1000
})
length_melted = length_df.melt(id_vars='chromosome', var_name='group', value_name='avg_length_kb')
length_df

# Step 2: Plot it
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=length_melted, x='chromosome', y='avg_length_kb', hue='group',
            palette=['lightcoral', 'steelblue'], ax=ax)
ax.set_xlabel('Chromosome')
ax.set_ylabel('Average Gene Length (kb)')
ax.set_title('Average Gene Length: All Genes vs Rare Disease Genes')
ax.legend(title='')
plt.tight_layout()
plt.show()
```

</details>

### Grouped Bar Chart Quick Reference

| Line | What it does |
|------|--------------|
| `df.melt(id_vars='x', var_name='group', value_name='y')` | Reshape data for grouped plotting |
| `sns.barplot(data=df, x='x', y='y', hue='group')` | Grouped bar chart |
| `palette=['coral', 'steelblue']` | Set colors for each group |
| `ax.legend(title='')` | Show legend without a title |

**Answer this question** (double-click to edit):

1. Are rare disease genes generally longer or shorter than the average gene? Why might that be?  
   *Your answer:*

---
## Part 3: Do More Genes Mean More Disease?

Some chromosomes have thousands of genes, others have hundreds. If disease genes were randomly distributed, chromosomes with more genes should have proportionally more disease genes.

**Question: Is the number of disease genes on a chromosome simply proportional to its total gene count — or are some chromosomes outliers?**

In [ ]:
# Step 1: Prepare the data — build a table with both counts per chromosome
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

total_per_chrom = genes[genes['chromosome'].isin(chrom_order)].groupby('chromosome').size()
rare_per_chrom = (
    merged.drop_duplicates(subset='gene_symbol')
    .query("chromosome in @chrom_order")
    .groupby('chromosome').size()
)

chrom_df = pd.DataFrame({
    'chromosome': chrom_order,
    'total_genes': total_per_chrom.reindex(chrom_order).values,
    'rare_disease_genes': rare_per_chrom.reindex(chrom_order).fillna(0).astype(int).values
})

print("Genes per chromosome:")
chrom_df

In [ ]:
# Step 2: Plot it
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=chrom_df, x='total_genes', y='rare_disease_genes', s=80, color='steelblue', ax=ax)

# Label each dot with the chromosome name
for _, row in chrom_df.iterrows():
    ax.annotate(row['chromosome'], (row['total_genes'], row['rare_disease_genes']),
                textcoords='offset points', xytext=(5, 5), fontsize=9)

ax.set_xlabel('Total Genes on Chromosome')
ax.set_ylabel('Rare Disease Genes on Chromosome')
ax.set_title('Total Genes vs Rare Disease Genes per Chromosome')
plt.tight_layout()
plt.show()

### Scatter Plot Quick Reference

| Line | What it does |
|------|--------------|
| `ax.scatter(x, y)` | Create a scatter plot |
| `s=80` | Dot size |
| `ax.annotate(text, (x, y))` | Label a dot |
| `xytext=(5, 5)` | Offset the label so it doesn’t sit on the dot |

### 🎯 Exercise 3: Interpret the Scatter Plot

Look at the scatter plot above and answer these questions (double-click to edit):

1. Is there a relationship between total genes and rare disease genes? Describe the pattern.  
   *Your answer:*

2. Which chromosome seems to have **more** rare disease genes than you'd expect based on its total gene count? (Look for dots that are higher than the trend.)  
   *Your answer:*

3. Which chromosome has **fewer** rare disease genes than expected?  
   *Your answer:*

4. Why might chromosome X be unusual? (Hint: think about X-linked genetic diseases.)  
   *Your answer:*

---
## Part 4: Do Disease Types Cluster on Specific Chromosomes?

Cancer genes, heart disease genes, and brain disease genes might not be spread randomly across the genome. Some chromosomes might be hotspots for certain disease types.

**Question: Are cancer-related genes concentrated on particular chromosomes?**

In [ ]:
# Step 1: Prepare the data
keyword = 'cancer'
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

category = merged[merged['disease'].str.contains(keyword, case=False)]
category_genes = category.drop_duplicates(subset='gene_symbol')
chrom_counts = (
    category_genes['chromosome']
    .value_counts()
    .reindex(chrom_order)
    .dropna()
    .reset_index()
)
chrom_counts.columns = ['chromosome', 'count']

print(f"Unique '{keyword}' disease genes: {len(category_genes)}")
print(f"\nGenes per chromosome:")
chrom_counts

In [ ]:
# Step 2: Plot it
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=chrom_counts, x='chromosome', y='count', color='indianred', ax=ax)
ax.set_xlabel('Chromosome')
ax.set_ylabel(f'Unique Genes linked to "{keyword}" diseases')
ax.set_title(f'Rare Disease Genes for "{keyword}" by Chromosome')
plt.tight_layout()
plt.show()

### 🎯 Exercise 4: Does the Pattern Hold for Other Disease Types?

Cancer genes cluster on certain chromosomes. **Is the pattern different for other disease categories — like heart, eye, or kidney diseases?**

Pick a different keyword and find out.

In [ ]:
# YOUR CODE HERE: Pick a keyword and create a chromosome bar chart
my_keyword = ''   # Fill in your keyword

if not my_keyword:
    raise ValueError("Please fill in your keyword before running this cell!")

# Step 1: Prepare the data
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']
category = merged[merged['disease'].str.contains(my_keyword, case=False)]
category_genes = category.drop_duplicates(subset='gene_symbol')
chrom_counts = (
    category_genes['chromosome']
    .value_counts()
    .reindex(chrom_order)
    .dropna()
    .reset_index()
)
chrom_counts.columns = ['chromosome', 'count']
print(f"Unique '{my_keyword}' disease genes: {len(category_genes)}")
chrom_counts

# Step 2: Plot it
# YOUR CODE HERE: use sns.barplot() — copy the pattern from the demo above



<details>
<summary>💡 Click here for the solution</summary>

```python
# Step 1: Prepare the data
my_keyword = 'cardiac'
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

category = merged[merged['disease'].str.contains(my_keyword, case=False)]
category_genes = category.drop_duplicates(subset='gene_symbol')
chrom_counts = (
    category_genes['chromosome']
    .value_counts()
    .reindex(chrom_order)
    .dropna()
    .reset_index()
)
chrom_counts.columns = ['chromosome', 'count']
chrom_counts

# Step 2: Plot it
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=chrom_counts, x='chromosome', y='count', color='indianred', ax=ax)
ax.set_xlabel('Chromosome')
ax.set_ylabel(f'Unique Genes linked to "{my_keyword}" diseases')
ax.set_title(f'Rare Disease Genes for "{my_keyword}" by Chromosome')
plt.tight_layout()
plt.show()
```

</details>

**Answer these questions** (double-click to edit):

1. Which keyword did you choose?  
   *Your answer:*

2. Which chromosome has the most genes for your category?  
   *Your answer:*

3. Are the results different from the cancer example? What might explain the differences?  
   *Your answer:*

---
## Part 5: What Does the "Typical" Disease Gene Look Like?

We know the *average* gene length, but averages can be misleading. Are most disease genes clustered around the average, or is there a wide spread with some extremely long and short genes?

**Question: How are rare disease gene lengths distributed compared to all genes?**

In [ ]:
# Step 1: Prepare the data
unique_disease_genes = merged.drop_duplicates(subset='gene_symbol')

# Build a combined DataFrame with a label column
all_df = pd.DataFrame({'length_kb': genes['length'] / 1000, 'group': 'All Genes'})
rare_df = pd.DataFrame({'length_kb': unique_disease_genes['length'].dropna() / 1000, 'group': 'Rare Disease Genes'})
length_combined = pd.concat([all_df, rare_df], ignore_index=True)

print("Gene length summary (in kb):")
print(f"\n  All genes:          median = {all_df['length_kb'].median():,.0f} kb, mean = {all_df['length_kb'].mean():,.0f} kb")
print(f"  Rare disease genes: median = {rare_df['length_kb'].median():,.0f} kb, mean = {rare_df['length_kb'].mean():,.0f} kb")
print(f"\n  Combined rows for plotting: {len(length_combined)}")

In [ ]:
# Step 2: Plot it
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=length_combined, x='length_kb', hue='group', bins=50,
             stat='density', common_norm=False, alpha=0.5,
             palette=['lightcoral', 'steelblue'], ax=ax)
ax.set_xlabel('Gene Length (kb)')
ax.set_ylabel('Density')
ax.set_title('Distribution of Gene Lengths: All Genes vs Rare Disease Genes')
ax.set_xlim(0, 500)
plt.tight_layout()
plt.show()

### Histogram Quick Reference

| Line | What it does |
|------|--------------|
| `sns.histplot(data=df, x='col', hue='group')` | Overlapping histograms by group |
| `bins=50` | Number of bins |
| `stat='density'` | Normalize so areas sum to 1 (fair comparison when group sizes differ) |
| `common_norm=False` | Normalize each group separately |
| `alpha=0.5` | Semi-transparent so both distributions are visible |
| `ax.set_xlim(0, 500)` | Zoom in on x-axis range |

### 🎯 Exercise 5: Do Different Disease Types Have Different Gene Sizes?

Cancer genes, cardiac genes, and neurological genes might have different size profiles.

**Question: Does your chosen disease category have longer or shorter genes than average?**

Pick a keyword and compare its gene length distribution to all genes.

In [ ]:
# YOUR CODE HERE: Histogram of gene lengths for a disease category



<details>
<summary>💡 Click here for the solution</summary>

```python
# Step 1: Prepare the data
my_keyword = 'cardiac'
my_category = merged[merged['disease'].str.contains(my_keyword, case=False)]
my_category_genes = my_category.drop_duplicates(subset='gene_symbol')

all_df = pd.DataFrame({'length_kb': genes['length'] / 1000, 'group': 'All Genes'})
my_df = pd.DataFrame({'length_kb': my_category_genes['length'].dropna() / 1000, 'group': f'"{my_keyword}" Genes'})
combined = pd.concat([all_df, my_df], ignore_index=True)

print(f"'{my_keyword}' genes: {len(my_category_genes)}, median length = {my_df['length_kb'].median():,.0f} kb")

# Step 2: Plot it
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=combined, x='length_kb', hue='group', bins=50,
             stat='density', common_norm=False, alpha=0.5,
             palette=['lightcoral', 'steelblue'], ax=ax)
ax.set_xlabel('Gene Length (kb)')
ax.set_ylabel('Density')
ax.set_title(f'Gene Length Distribution: All Genes vs "{my_keyword}" Disease Genes')
ax.set_xlim(0, 500)
plt.tight_layout()
plt.show()
```

</details>

**Answer these questions** (double-click to edit):

1. Are most rare disease genes short or long?  
   *Your answer:*

2. Does the distribution for your disease category look different from the overall distribution?  
   *Your answer:*

---
## Part 6: Do Different Gene Types Have Different Sizes?

Not all genes are the same type. Protein-coding genes make proteins; lncRNAs regulate other genes; pseudogenes are broken copies. These different **biotypes** might have very different size characteristics.

**Question: How does gene length vary across different types of rare disease genes?**

In [ ]:
# Step 1: Prepare the data
unique_disease_genes = merged.drop_duplicates(subset='gene_symbol').copy()
top5_biotypes = unique_disease_genes['biotype'].value_counts().head(5)

print("Top 5 biotypes of rare disease genes:")
print(top5_biotypes)

plot_data = unique_disease_genes[unique_disease_genes['biotype'].isin(top5_biotypes.index)]
print(f"\nFiltered to {len(plot_data)} genes for plotting")

In [ ]:
# Step 2: Plot it
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=plot_data, x='biotype', y='length', order=top5_biotypes.index, ax=ax)
ax.set_ylabel('Gene Length (base pairs)')
ax.set_xlabel('Biotype')
ax.set_title('Gene Length Distribution by Biotype (Rare Disease Genes)')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

### Box Plot Quick Reference

| Line | What it does |
|------|--------------|
| `sns.boxplot(data=df, x='col1', y='col2')` | Create a box plot grouped by `x` |
| `order=[...]` | Control the order of categories on x-axis |
| `ax.set_yscale('log')` | Use log scale (helpful when values span many orders of magnitude) |

### 🎯 Exercise 6: How Variable Are Gene Sizes Across Chromosomes?

Some chromosomes might have consistently sized disease genes, while others might have a wide range from tiny to enormous.

**Question: Which chromosomes have the most variable rare disease gene sizes?**

Create a box plot to find out.

In [ ]:
# YOUR CODE HERE: Box plot of gene length by chromosome



<details>
<summary>💡 Click here for the solution</summary>

```python
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=unique_disease_genes, x='chromosome', y='length',
            order=chrom_order, ax=ax)
ax.set_ylabel('Gene Length (base pairs)')
ax.set_xlabel('Chromosome')
ax.set_title('Gene Length Distribution by Chromosome (Rare Disease Genes)')
ax.set_yscale('log')
plt.tight_layout()
plt.show()
```

</details>

**Answer these questions** (double-click to edit):

1. Which chromosome has the widest spread of gene lengths? Which has the narrowest?  
   *Your answer:*

2. Are there many outliers (dots)? What does an outlier mean in this context?  
   *Your answer:*

---
## Part 7: Can We See All the Patterns at Once?

So far we've looked at one disease category or one chromosome at a time. But what if we want to see **all disease categories across all chromosomes** in a single view?

**Question: Which chromosomes are hotspots for which disease types?**

A heatmap lets us answer this by showing the full picture in one chart.

In [ ]:
# Step 1: Prepare the data — build a table of disease categories × chromosomes
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

categories = {
    'Cancer': 'cancer|tumor|carcinoma|lymphoma|leukemia',
    'Heart': 'heart|cardiac|cardiomyopathy',
    'Brain': 'brain|neuro|epilepsy|ataxia|neuropathy',
    'Eye': 'eye|retinal|blindness|optic|macular',
    'Muscle': 'muscle|muscular|myopathy|dystrophy',
    'Kidney': 'kidney|renal|nephro',
    'Immune': 'immune|immunodeficiency',
    'Bone': 'bone|osteo|skeletal',
}

heatmap_data = {}
for cat, pattern in categories.items():
    filtered = merged[merged['disease'].str.contains(pattern, case=False)]
    gene_counts = filtered.drop_duplicates(subset='gene_symbol')
    counts = gene_counts['chromosome'].value_counts().reindex(chrom_order).fillna(0)
    heatmap_data[cat] = counts

heatmap_df = pd.DataFrame(heatmap_data).T

print("Disease categories × chromosomes (unique gene counts):")
heatmap_df

In [ ]:
# Step 2: Plot it
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heatmap_df, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_xlabel('Chromosome')
ax.set_ylabel('Disease Category')
ax.set_title('Rare Disease Genes: Disease Categories by Chromosome')
plt.tight_layout()
plt.show()

### 🎯 Exercise 7: Interpret the Heatmap

Look at the heatmap above and answer these questions (double-click to edit):

1. Which chromosome is a "hotspot" for cancer-related genes?  
   *Your answer:*

2. Which disease category has the most genes on chromosome X?  
   *Your answer:*

3. Are there any chromosomes that have very few disease genes across all categories? Which ones?  
   *Your answer:*

4. Pick one bright cell (high value) in the heatmap. Why might that chromosome have many genes for that disease type?  
   *Your answer:*

---
## Part 8: What's Your Question?

You've seen how visualizations can answer biological questions. Now it's your turn to ask one.

### 🎯 Exercise 8: Investigate and Visualize

Pick **one** question below (or make up your own) and answer it with code and a chart:

1. **Which chromosome has the longest rare disease genes?**  
   → Group by chromosome, calculate average gene length, make a bar chart

2. **Do forward-strand and reverse-strand genes have different disease counts?**  
   → Filter by strand (1 or -1), compare disease counts with a grouped bar chart

3. **Compare two disease keywords — which chromosomes differ?**  
   → Filter merged for two keywords, count by chromosome, plot side by side

4. **Your own question!**  
   → Any question you can answer with the merged dataset

In [ ]:
# YOUR CODE HERE: Investigate your chosen question


In [ ]:
# YOUR CODE HERE: Create a visualization for your question


### 🎯 Exercise 9: Written Summary

In 3–5 sentences, summarize what you found in Exercise 8.  
(Double-click to edit)

- What question did you pick and why?
- What did your chart show?
- What was the most surprising finding?

*Your answer:*

---
## ⭐ Extension Challenge (Optional)

### Stacked Bar Chart: Multiple Disease Categories by Chromosome

Create a **stacked bar chart** showing how many genes from different disease categories are on each chromosome.  
Pick 3–4 keywords and stack them.

*Hint: Use `ax.bar()` multiple times with the `bottom` parameter to stack bars.*

In [ ]:
# YOUR CODE HERE: Stacked bar chart of disease categories by chromosome
#
# Template:
#   chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']
#   keywords = ['cancer', 'cardiac', 'renal']   # pick your own
#
#   fig, ax = plt.subplots(figsize=(14, 6))
#   bottom = np.zeros(len(chrom_order))
#
#   for kw in keywords:
#       filtered = merged[merged['disease'].str.contains(kw, case=False)]
#       counts = filtered.drop_duplicates(subset='gene_symbol')['chromosome'].value_counts()
#       counts = counts.reindex(chrom_order).fillna(0).values
#       ax.bar(chrom_order, counts, bottom=bottom, label=kw)
#       bottom += counts
#
#   ax.set_xlabel('Chromosome')
#   ax.set_ylabel('Unique Disease Genes')
#   ax.set_title('Disease Categories by Chromosome')
#   ax.legend()
#   plt.tight_layout()
#   plt.show()


---
## Quick Reference: All Chart Types

| Chart | When to use | Key function |
|-------|------------|-------------|
| Bar chart | Compare counts across categories | `sns.barplot(data=df, x='col', y='col')` |
| Scatter plot | Show relationship between two numbers | `sns.scatterplot(data=df, x='col', y='col')` |
| Grouped bar | Compare two groups side by side | `sns.barplot(data=melted_df, x='col', y='col', hue='group')` |
| Histogram | Show distribution of values | `sns.histplot(data, bins=50)` |
| Box plot | Compare distributions across groups | `sns.boxplot(data=df, x='col', y='col')` |
| Heatmap | Show values across two dimensions | `sns.heatmap(df, annot=True, cmap='YlOrRd')` |
| Stacked bar | Show composition across categories | `ax.bar(x, y, bottom=prev)` |

### Common Styling

| Line | What it does |
|------|--------------|
| `fig, ax = plt.subplots(figsize=(10, 6))` | Set figure size |
| `ax.set_xlabel('...')` | Label x-axis |
| `ax.set_ylabel('...')` | Label y-axis |
| `ax.set_title('...')` | Add title |
| `ax.legend()` | Show legend |
| `ax.set_yscale('log')` | Log scale for large value ranges |
| `alpha=0.5` | Semi-transparent (for overlapping plots) |
| `plt.tight_layout()` | Prevent labels from being cut off |
| `plt.show()` | Display the chart |